<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/Console.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from google.colab import files

warnings.filterwarnings("ignore")

# ==============================================================================
# 0. 全局交易設定與標的池 (已修正 6284.TWO 上櫃代號)
# ==============================================================================
TOTAL_PORTFOLIO_CAPITAL = (
    10000000  # 擬投入組合總資金 (例如：NTD 10,000,000)
)

custom_ticker_map = {
    "0050.TW": "元大台灣50",
    "0056.TW": "元大高股息",
    "00878.TW": "國泰永續高股息",
    "00770.TW": "國泰北美科技",
    "00981A.TW": "統一台股增長主動式",
    "SPCX": "SPACs ETF",
    "SOXX": "iShares半導體ETF",
    "SMH": "VanEck半導體ETF",
    "AAPL": "Apple 蘋果",
    "GOOG": "Google / Alphabet",
    "META": "Meta",
    "MSFT": "Microsoft 微軟",
    "NVDA": "NVIDIA 輝達",
    "TSM": "台積電 ADR",
    "TSLA": "Tesla 特斯拉",
    "ENTG": "Entegris 英特格",
    "SMR": "NuScale Power 小型核反應爐",
    "BE": "Bloom Energy 燃料電池",
    "JNJ": "Johnson & Johnson 嬌生",
    "ASML": "ASML 艾司摩爾",
    "AMAT": "Applied Materials 應用材料",
    "LRCX": "Lam Research 柯林研發",
    "KLAC": "KLA 科磊",
    "AMD": "AMD 超微",
    "AVGO": "Broadcom 博通",
    "QCOM": "Qualcomm 高通",
    "INTC": "Intel 英特爾",
    "MU": "Micron 鎂光",
    "TXN": "Texas Instruments 德州儀器",
    "ARM": "ARM 晶心/安謀",
    "MRVL": "Marvell 邁威爾",
    "ADI": "Analog Devices 亞德諾",
    "MPWR": "Monolithic Power 芯源系統",
    "ON": "ON Semiconductor 安森美",
    "SWKS": "Skyworks 思佳訊",
    "QRVO": "Qorvo 威訊",
    "TER": "Teradyne 泰瑞達",
    "MKSI": "MKS Instruments",
    "PANW": "Palo Alto Networks",
    "CRWD": "CrowdStrike",
    "FTNT": "Fortinet",
    "NET": "Cloudflare",
    "ZS": "Zscaler",
    "OKTA": "Okta",
    "S": "SentinelOne",
    "GEN": "Gen Digital",
    "RPD": "Rapid7",
    "CBRS": "CyberArk",
    "2471.TW": "資通",
    "2480.TW": "敦陽科",
    "3029.TW": "零壹",
    "6214.TW": "精誠",
    "3130.TW": "一零四",
    "2427.TW": "三商電",
    "3027.TW": "盛達",
    "5203.TW": "訊連",
    "5471.TW": "松翰",
    "5410.TWO": "國統",
    "6183.TW": "關貿",
    "6203.TWO": "海韻電",
    "6210.TWO": "慶生",
    "6593.TWO": "台灣銘板",
    "6689.TW": "伊雲谷",
    "6690.TWO": "安碁資訊",
    "6752.TWO": "睿嘉",
    "6763.TWO": "綠界科技",
    "6865.TWO": "偉康科技",
    "6874.TWO": "倍力",
    "6928.TW": "全達",
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微 MSI",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "2353.TW": "宏碁",
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    "3035.TW": "智原",
    "6643.TWO": "M31",
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創唯",
    "6756.TW": "威鋒電子",
    "2342.TW": "茂矽",
    "6770.TW": "力積電",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "钛昇",
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    "2404.TW": "漢唐",
    "1773.TW": "勝一",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "6613.TWO": "朋億*",
    "4755.TW": "三福化",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "5536.TWO": "聖暉*",
    "3644.TWO": "凌嘉科",
    "7769.TW": "鴻勁",
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜晶",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    "6715.TW": "嘉基",
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必股",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    "8043.TWO": "蜜望實",
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    "2395.TW": "研華",
    "6166.TW": "凌華",
    "8050.TWO": "廣積",
    "3556.TWO": "禾瑞亞",
    "2414.TW": "精技",
    "6414.TW": "樺漢",
    "3022.TW": "威強電",
    "2397.TW": "友通",
    "5314.TWO": "世紀",
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    "3481.TW": "群創",
    "2409.TW": "友達",
    "3008.TW": "大立光",
    "4915.TW": "先進光",
    "5288.TW": "匯鑽科",
    "2393.TW": "億光",
    "2201.TW": "裕隆",
    "2204.TW": "中華",
    "2206.TW": "三陽工業",
    "1536.TW": "和大",
    "2231.TW": "聯嘉",
    "3552.TWO": "同致",
    "6279.TWO": "胡連",
    "2603.TW": "長榮",
    "2609.TW": "陽明",
    "2615.TW": "萬海",
    "2605.TW": "新興",
    "2606.TW": "裕民",
    "2612.TW": "中航",
    "2617.TW": "台航",
    "2637.TW": "慧洋-KY",
    "2641.TWO": "正德",
    "5608.TW": "四維航",
    "2610.TW": "華航",
    "2618.TW": "長榮航",
    "2630.TW": "亞航",
    "5603.TWO": "陸海",
    "2607.TW": "勞運",
    "2608.TW": "嘉里大榮",
    "2611.TW": "志信",
    "2613.TW": "中櫃",
    "2636.TW": "台驊投控",
    "2642.TW": "宅配通",
    "2633.TW": "台灣高鐵",
    "5607.TW": "遠雄港",
    "5609.TWO": "中菲行",
    "8367.TW": "建新國際",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "1303.TW": "南亞",
    "2465.TW": "麗臺",
    "8163.TW": "達方",
    "3042.TW": "晶技",
    "8182.TWO": "加高",
    "3229.TW": "泰藝",
    "3308.TW": "聯傑",
    "6284.TWO": "佳邦",  # 修正為 .TWO 上櫃代號
    "2484.TW": "希華",
    "8088.TWO": "華信科",
}

ticker_to_name = custom_ticker_map
all_tickers = list(ticker_to_name.keys())


def classify_sector(ticker):
    t = str(ticker)
    name = ticker_to_name.get(t, "")
    if any(
        k in t
        for k in [
            "0050",
            "0056",
            "00878",
            "00770",
            "00981",
            "SPCX",
            "SOXX",
            "SMH",
        ]
    ):
        return "ETF資產配置"
    if not t.endswith(".TW") and not t.endswith(".TWO"):
        if any(
            sec in t
            for sec in [
                "NVDA",
                "AAPL",
                "GOOG",
                "META",
                "MSFT",
                "TSM",
                "ASML",
                "AMAT",
                "LRCX",
                "KLAC",
                "AMD",
                "AVGO",
                "QCOM",
                "INTC",
                "MU",
                "TXN",
                "ARM",
                "MRVL",
                "ADI",
                "MPWR",
                "ON",
                "SWKS",
                "QRVO",
                "TER",
                "MKSI",
                "PANW",
                "CRWD",
                "FTNT",
                "NET",
                "ZS",
                "OKTA",
                "S",
                "GEN",
                "RPD",
                "CBRS",
            ]
        ):
            return "美股科技/半導體/資安"
        return "美股其他"
    else:
        if any(
            w in name
            for w in [
                "航",
                "運",
                "高鐵",
                "統一",
                "大成",
                "卜蜂",
                "全家",
                "南亞",
                "金融",
                "第一金",
                "合庫金",
            ]
        ):
            return "台股傳產/金融/航運"
        return "台股電子/半導體/供應鏈"


ticker_to_sector = {t: classify_sector(t) for t in all_tickers}


# ==============================================================================
# 1. 模擬歷史預測與風控數據
# ==============================================================================
def simulate_base_prediction_pipeline(tickers):
    np.random.seed(42)
    n = len(tickers)

    p_pred_raw = np.random.uniform(0.40, 0.85, n)
    win_rate = p_pred_raw * np.random.uniform(0.85, 0.98, n)
    avg_win = np.random.uniform(0.03, 0.08, n)
    avg_loss = np.random.uniform(0.015, 0.035, n)

    eval_score = (win_rate * avg_win) / (avg_loss + 1e-5) * 10
    prob_threshold = 0.55

    is_valid_signal = np.where(p_pred_raw >= prob_threshold, "有效交易", "濾除觀望")
    base_weight = np.where(is_valid_signal == "有效交易", p_pred_raw * 0.2, 0.0)
    sector_cap = 0.25

    records = []
    for i, t in enumerate(tickers):
        sec = ticker_to_sector.get(t, "未知產業")
        name = ticker_to_name.get(t, t)

        rec = {
            "股票代號": t,
            "股票名稱": name,
            "產業分類": sec,
            "p_pred_raw": round(p_pred_raw[i], 4),
            "模型判定看漲機率": f"{p_pred_raw[i]*100:.2f}%",
            "機率閾值(濾除噪訊)": f"{prob_threshold*100:.1f}%",
            "勝率(Win Rate)": round(win_rate[i], 4),
            "平均獲利(Avg Win)": round(avg_win[i], 4),
            "平均損失(Avg Loss)": round(avg_loss[i], 4),
            "評估分數": round(eval_score[i], 2),
            "專業優化訊號有效性": is_valid_signal[i],
            "原始建議部位": f"{base_weight[i]*100:.2f}%",
            "單一產業上限": f"{sector_cap*100:.0f}%",
            "勝率_num": win_rate[i],
            "avg_win": avg_win[i],
            "avg_loss": avg_loss[i],
            "eval_score": eval_score[i],
            "base_weight": base_weight[i],
        }
        records.append(rec)

    df = pd.DataFrame(records)

    df["sector_total_weight"] = df.groupby("產業分類")["base_weight"].transform(
        "sum"
    )
    df["sector_scale"] = np.where(
        df["sector_total_weight"] > sector_cap,
        sector_cap / (df["sector_total_weight"] + 1e-9),
        1.0,
    )
    df["rebalanced_weight"] = df["base_weight"] * df["sector_scale"]

    df["風控頂格再平衡建議部位比率"] = (df["rebalanced_weight"] * 100).map(
        "{:.2f}%".format
    )
    df["風控頂格再平衡建議部位_num"] = df["rebalanced_weight"] * 100
    df["風控狀態描述"] = np.where(
        df["sector_scale"] < 1.0,
        "觸及產業上限(已等比縮減)",
        "風控正常(未觸及上限)",
    )

    df["5日3%回報率模型勝率"] = df["勝率(Win Rate)"].map("{:.2f}%".format)
    df["5日平均獲利幅度"] = df["平均獲利(Avg Win)"].map("{:.2f}%".format)
    df["5日平均虧損幅度"] = df["平均損失(Avg Loss)"].map("{:.2f}%".format)
    df["綜合期望值評估分數"] = df["評估分數"]

    cols_order = [
        "股票代號",
        "股票名稱",
        "產業分類",
        "模型判定看漲機率",
        "機率閾值(濾除噪訊)",
        "專業優化訊號有效性",
        "5日3%回報率模型勝率",
        "5日平均獲利幅度",
        "5日平均虧損幅度",
        "綜合期望值評估分數",
        "原始建議部位",
        "單一產業上限",
        "風控頂格再平衡建議部位比率",
        "風控狀態描述",
        "p_pred_raw",
        "勝率_num",
        "avg_win",
        "avg_loss",
        "eval_score",
        "base_weight",
        "風控頂格再平衡建議部位_num",
    ]
    return df[cols_order]


# ==============================================================================
# 2. 交易員與 PM 級別「可執行交易報表」優化模組 (加入個別安全容錯下載)
# ==============================================================================
def enrich_trader_execution_sheet(df_res, total_portfolio_capital=10000000):
    df_trade = df_res.copy()

    print(
        f"🔍 正在一次性向量化拉取 {len(df_trade)} 隻標的最新行情與波動度數據 (yfinance Batch Download)..."
    )

    tickers_list = df_trade["股票代號"].tolist()

    try:
        data = yf.download(
            tickers_list,
            period="1mo",
            progress=False,
            group_by="ticker",
            auto_adjust=True,
        )
    except Exception as e:
        print(f"⚠️ 網路下載數據異常: {e}")
        data = None

    ref_prices = []
    atrs = []
    advs = []

    for ticker in tickers_list:
        default_price = 500.0 if (".TW" in ticker or ".TWO" in ticker) else 150.0
        default_atr = 10.0
        default_adv = 500000000.0

        try:
            df_t = None
            if data is not None:
                # 兼容多重 DataFrame 結構判斷
                if len(tickers_list) == 1:
                    df_t = data.dropna()
                elif ticker in data:
                    df_t = data[ticker].dropna()

            if df_t is not None and not df_t.empty and len(df_t) >= 5:
                latest_close = float(df_t["Close"].iloc[-1])
                tr = np.maximum(
                    df_t["High"] - df_t["Low"],
                    np.maximum(
                        abs(df_t["High"] - df_t["Close"].shift(1)),
                        abs(df_t["Low"] - df_t["Close"].shift(1)),
                    ),
                )
                atr = float(tr.rolling(min(14, len(tr))).mean().iloc[-1])
                adv = float((df_t["Close"] * df_t["Volume"]).tail(20).mean())
            else:
                latest_close, atr, adv = default_price, default_atr, default_adv
        except Exception:
            latest_close, atr, adv = default_price, default_atr, default_adv

        ref_prices.append(latest_close)
        atrs.append(atr)
        advs.append(adv)

    df_trade["最新參考價"] = ref_prices
    df_trade["ATR_14"] = atrs
    df_trade["20日均成交額"] = advs

    rebalance_pct = df_trade["風控頂格再平衡建議部位_num"] / 100.0
    conditions = [
        (df_trade["專業優化訊號有效性"] == "有效交易") & (rebalance_pct >= 0.05),
        (df_trade["專業優化訊號有效性"] == "有效交易") & (rebalance_pct > 0.0),
    ]
    choices = ["STRONG_BUY", "BUY_NEW"]
    df_trade["建議交易動作"] = np.select(conditions, choices, default="NO_ACTION")
    df_trade["目標分配金額"] = total_portfolio_capital * rebalance_pct

    def calc_shares(row):
        price = row["最新參考價"]
        capital = row["目標分配金額"]
        ticker = str(row["股票代號"])

        if pd.isna(price) or price <= 0 or capital <= 0:
            return "0 股"

        raw_shares = capital / price
        if ".TW" in ticker or ".TWO" in ticker:
            lots = int(raw_shares // 1000)
            odd_shares = int(raw_shares % 1000)
            if lots > 0:
                return (
                    f"{lots} 張 ({odd_shares} 股)"
                    if odd_shares > 0
                    else f"{lots} 張"
                )
            else:
                return f"{odd_shares} 股"
        else:
            return f"{int(raw_shares)} 股"

    df_trade["預估下單數量"] = df_trade.apply(calc_shares, axis=1)
    df_trade["建議停損價(SL)"] = (
        df_trade["最新參考價"] - (1.5 * df_trade["ATR_14"])
    ).round(2)
    df_trade["建議停利價(TP)"] = (
        df_trade["最新參考價"] * (1 + df_trade["avg_win"])
    ).round(2)

    df_trade["日均成交額占比(%)"] = (
        (df_trade["目標分配金額"] / (df_trade["20日均成交額"] + 1e-9)) * 100
    ).round(2)
    df_trade["流動性風險評估"] = np.where(
        df_trade["日均成交額占比(%)"] > 5.0,
        "高衝擊(建議分批)",
        "良好",
    )

    df_trade["目標分配金額(NTD/USD)"] = df_trade["目標分配金額"].apply(
        lambda x: f"${x:,.0f}"
    )
    df_trade["最新參考價_fmt"] = df_trade["最新參考價"].round(2)

    df_trade.sort_values(
        by=["風控頂格再平衡建議部位_num", "p_pred_raw"],
        ascending=[False, False],
        inplace=True,
    )

    filename = "Portfolio_Execution_Order_Sheet.xlsx"
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        execution_mask = df_trade["建議交易動作"].isin(
            ["STRONG_BUY", "BUY_NEW"]
        )
        execution_cols = [
            "股票代號",
            "股票名稱",
            "產業分類",
            "建議交易動作",
            "最新參考價_fmt",
            "目標分配金額(NTD/USD)",
            "預估下單數量",
            "建議停損價(SL)",
            "建議停利價(TP)",
            "風控頂格再平衡建議部位比率",
            "流動性風險評估",
        ]
        execution_rename = {"最新參考價_fmt": "最新參考價"}

        df_exec = df_trade[execution_mask][execution_cols].rename(
            columns=execution_rename
        )

        df_top10 = df_exec.head(10)
        df_top10.to_excel(writer, sheet_name="Top10_Orders", index=False)
        df_exec.to_excel(
            writer, sheet_name="Trader_Execution_Orders", index=False
        )
        df_trade.to_excel(writer, sheet_name="PM_Full_Analysis", index=False)

    print(f"✅ 已成功整合更新清單！可執行報表已匯出至：{filename}\n")
    return df_trade, df_top10, filename


# ==============================================================================
# 3. 主執行流程 (Main Pipeline)
# ==============================================================================
if __name__ == "__main__":
    print("🚀 啟動量化預測與交易報表生成系統...")

    df_base_result = simulate_base_prediction_pipeline(all_tickers)
    df_final_trade, df_top10, excel_filename = enrich_trader_execution_sheet(
        df_base_result, total_portfolio_capital=TOTAL_PORTFOLIO_CAPITAL
    )

    print("=" * 100)
    print("📋【交易員開盤可執行下單清單 (Top 10)】")
    print("=" * 100)
    print(df_top10.to_string(index=False))
    print("=" * 100)

    print(f"📥 修正完成！正在啟動 Google Colab 檔案下載：{excel_filename} ...")
    files.download(excel_filename)


🚀 啟動量化預測與交易報表生成系統...
🔍 正在一次性向量化拉取 340 隻標的最新行情與波動度數據 (yfinance Batch Download)...
✅ 已成功整合更新清單！可執行報表已匯出至：Portfolio_Execution_Order_Sheet.xlsx

📋【交易員開盤可執行下單清單 (Top 10)】
    股票代號                 股票名稱       產業分類     建議交易動作  最新參考價 目標分配金額(NTD/USD)       預估下單數量  建議停損價(SL)  建議停利價(TP) 風控頂格再平衡建議部位比率 流動性風險評估
      BE    Bloom Energy 燃料電池       美股其他 STRONG_BUY 275.75      $1,272,281       4613 股     250.33     292.04        12.72%      良好
     JNJ Johnson & Johnson 嬌生       美股其他 STRONG_BUY 265.58      $1,188,751       4476 股     257.70     275.60        11.89%      良好
 0056.TW                元大高股息    ETF資產配置 STRONG_BUY  55.40        $577,291 10 張 (420 股)      54.21      57.78         5.77%      良好
     SMH         VanEck半導體ETF    ETF資產配置 STRONG_BUY 568.53        $550,762        968 股     547.86     593.54         5.51%      良好
00878.TW              國泰永續高股息    ETF資產配置 STRONG_BUY  34.23        $508,654 14 張 (859 股)      33.64      35.55         5.09%      良好
00770.TW               國泰北美科技    ETF資產配置  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>